<a href="https://colab.research.google.com/github/MoEissa140/FlyRank-Intern/blob/main/work/notebooks/w06_validation_audit.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-09 — Validation and Research Claim Audit

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/MoEissa140/FlyRank-Intern/blob/main/work/notebooks/w06_validation_audit.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Two paper findings + my methodology questions

**Finding: The Freshness Multiplier.** The paper reports that among pages older than a year, the cohort refreshed in the last 30 days jumped in health score and saw roughly a 52x lift in impressions compared to pages last touched 181-360 days prior, and frames the 31-90 day freshness window as the strongest stable growth signal at a 5.43:1 growth-to-decline ratio.

**My methodology question:** the report notes that content age confounds model comparisons in general — so for this specific finding, was "refreshed" assigned independently of prior performance, or could pages that were already recovering get refreshed *because* someone noticed early growth, then get credit for a lift that would have happened anyway? Put plainly: does the refresh happen before the recovery is visible, or could refresh timing itself be responding to an early signal of recovery? The paper doesn't specify whether refresh timing was randomized or reviewer-selected, and if it's the latter, the 52x figure could be inflated by selection rather than the refresh action itself.

---

**Finding: Growth Prediction model.** The paper reports a Random Forest-style model trained on ~96.6K pages that classifies growing vs. declining pages, reaching roughly 90% accuracy on same-brand held-out pages but dropping to about 75% on entirely unseen brands, with "days visible" as the top predictor.

**My methodology question:** the label used is a trend-direction bucket (up more than 10%, down more than 10%, comparing the last 30 days to the prior 30 days) rather than a truly future-looking outcome relative to a fixed decision point — this is close to the same "proxy label" pattern flagged in the starter notebook (`trend_direction`) rather than a strict past-feature-window → future-target-window design. Given the same-brand vs. new-brand accuracy gap (90% vs 75%), was the split grouped by brand for both evaluations, or could same-brand pages from the same time window have leaked temporal/seasonal signal into training that a genuinely time-aware split would have caught?

## 2. My model under an honest split (before/after)

*Re-run your Week-5 model under a grouped or time-aware split. Show both numbers.*

In [ ]:
!pip install -q huggingface_hub duckdb
import duckdb, os, pandas as pd
from google.colab import userdata
os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")

con = duckdb.connect()
con.execute("INSTALL httpfs; LOAD httpfs;")
con.execute(f"""
CREATE SECRET hf_token (
    TYPE huggingface,
    TOKEN '{os.environ["HF_TOKEN"]}'
);
""")

base = "hf://datasets/FlyRank/internship-warehouse"

feature_vector = con.sql(f"""
WITH feat_window AS (
    SELECT
        content_hash_id,
        client_hash_id,
        AVG(gsc_impressions) AS avg_impressions,
        AVG(gsc_clicks) AS avg_clicks,
        AVG(gsc_avg_position) AS avg_position,
        SUM(ga4_engaged_sessions) * 1.0 / NULLIF(SUM(ga4_sessions), 0) AS engagement_rate,
        SUM(scroll_events) * 1.0 / NULLIF(SUM(ga4_pageviews), 0) AS scroll_rate
    FROM read_parquet('{base}/fact_content_daily_performance/month=2026-03/data_0.parquet')
    GROUP BY content_hash_id, client_hash_id
),
target_window AS (
    SELECT
        content_hash_id,
        AVG(gsc_impressions) AS avg_impressions_next
    FROM read_parquet('{base}/fact_content_daily_performance/month=2026-04/data_0.parquet')
    GROUP BY content_hash_id
)
SELECT
    f.*,
    CASE WHEN t.avg_impressions_next < f.avg_impressions THEN 1 ELSE 0 END AS is_declining_label
FROM feat_window f
JOIN target_window t USING (content_hash_id)
WHERE f.avg_impressions > 0
""").df()

feature_vector["engagement_rate"] = feature_vector["engagement_rate"].fillna(0)
feature_vector["scroll_rate"] = feature_vector["scroll_rate"].fillna(0)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

In [ ]:
from sklearn.model_selection import GroupKFold

# Reuse feature_vector from w03_feature_leakage_check.ipynb (Mar features -> Apr label)
X_cols = ["avg_impressions","avg_clicks","avg_position","engagement_rate","scroll_rate"]
X = feature_vector[X_cols].fillna(0)
y = feature_vector["is_declining_label"]
groups = feature_vector["client_hash_id"]

gkf = GroupKFold(n_splits=5)

In [ ]:
from sklearn.model_selection import train_test_split, GroupKFold
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score

# "Before" — naive random split (what a less careful analysis might do)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
naive_model = RandomForestClassifier(n_estimators=200, random_state=42).fit(X_train, y_train)
naive_auc = roc_auc_score(y_test, naive_model.predict_proba(X_test)[:,1])

# "After" — grouped by client (same as w05)
gkf = GroupKFold(n_splits=5)
grouped_aucs = []
for train_idx, test_idx in gkf.split(X, y, groups):
    m = RandomForestClassifier(n_estimators=200, random_state=42).fit(X.iloc[train_idx], y.iloc[train_idx])
    grouped_aucs.append(roc_auc_score(y.iloc[test_idx], m.predict_proba(X.iloc[test_idx])[:,1]))
grouped_auc = sum(grouped_aucs)/len(grouped_aucs)

print(f"Naive random split AUC: {naive_auc:.3f}")
print(f"Grouped (client-holdout) AUC: {grouped_auc:.3f}")

Naive random split AUC: 0.588
Grouped (client-holdout) AUC: 0.510


| Split | AUC |
|---|---|
| Naive random split | 0.588 |
| Grouped (client-holdout) | 0.510 |

The naive split scores notably higher (0.588) than the grouped, client-holdout split (0.510) — a gap of 0.078. This is exactly the memorization effect flagged in ML-08: with a random split, pages from the same client can appear in both train and test, letting the model partially learn client-specific quirks (baseline traffic level, industry norms, tracking history) rather than a signal that generalizes to clients it has never seen. The grouped AUC (0.510) is the honest number — barely above chance — and is the one that should be reported and trusted going forward. This also matches the paper's own finding that its Growth Prediction model dropped from ~90% (same-brand) to ~75% (unseen brands): the same directional effect, at a much larger scale.

## 3. Leakage audit

*The same hunt from Week 3, on your final feature set.*

In [ ]:
# Confirm no product flags or future-window columns are present
assert not {"health_score","priority_score","action_type","avg_impressions_next"} & set(X_cols)

# Re-verify the deliberate leak-and-remove test still holds on the final set
feature_vector["leaky_test"] = feature_vector["avg_impressions_next"] if "avg_impressions_next" in feature_vector.columns else None
if feature_vector["leaky_test"].notna().any():
    leak_auc = honest_auc(feature_vector, X_cols + ["leaky_test"])
    print("Leak still detectable:", leak_auc)
feature_vector = feature_vector.drop(columns=["leaky_test"], errors="ignore")

Final feature set: `avg_impressions`, `avg_clicks`, `avg_position`, `engagement_rate`, `scroll_rate` — all confirmed knowable before the decision point (Mar features → Apr label). No product-decision flags (`health_score`, `priority_score`, `action_type`) are present in this schema (verified absent via `DESCRIBE` in ML-04). The deliberate leak-and-remove test from ML-05 confirmed that injecting a label-derived column (`avg_impressions_next`) inflates AUC sharply — that column is not present in the final feature set used here. Group-based split by `client_hash_id` (Section 2) prevents same-client leakage across train/test, and the resulting drop from 0.588 to 0.510 confirms that some of the naive split's apparent performance was leakage-adjacent (client memorization), not real generalizable signal.

## 4. Claim rewrite

**Original (too strong):** "The model proves that position and impressions are the true drivers of content decline, and this ranking can be trusted to prioritize review."

**Rewritten (safe language):** "In this observational sample, `avg_position` and `avg_impressions` were the two features the model relied on most heavily when ranking pages by predicted decline risk. Under an honest, client-grouped validation split, the model's AUC (0.510) is only marginally better than chance — this is a weak, directional decision-support signal at best, not proof of a reliable predictive relationship, and it reflects only what this dataset, this label definition, and this specific model configuration surfaced."

## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.